# 066 — Embeddings semánticos y similitud

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Hipótesis distribucional:** palabras con contextos similares significan cosas similares.
**word2vec skip-gram** la materializa: cada palabra central se entrena para dar producto
punto alto con sus contextos reales (`log σ(v_o·v_c)`) y bajo con k negativos aleatorios.
Emergen regularidades direccionales: `rey − hombre + mujer ≈ reina` (aproximado, no
exacto).

**Similitud coseno:** `cos(u,v) = u·v / (‖u‖‖v‖)` compara direcciones ignorando la
magnitud (que correlaciona con frecuencia, no con significado). Con vectores normalizados
equivale al producto punto.

**Límites:** los embeddings estáticos colapsan la polisemia ("banco"); los contextuales
(BERT, Sentence-BERT) dan un vector por token/frase. Y todos heredan los **sesgos** del
corpus (Bolukbasi 2016: `programador − hombre + mujer ≈ ama de casa`): la geometría
destila también los estereotipos del texto.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** `‖u‖ = √(4+4+1) = 3`, `‖v‖ = 3`. `u·v = 2+4+2 = 8` →
`cos = 8/9 ≈ 0.889`. Con `w = −u`: `cos(u, w) = −9/9 = −1`: dirección exactamente
opuesta — en un espacio semántico real casi nunca aparece, porque "opuesto en todo" no es
como se organizan los contextos.

**Ejercicio 2.** `cos(a,b) = 20/(√2·√200) = 20/20 = 1.0` (misma dirección);
`cos(a,c) = 1/(√2·1) ≈ 0.707`. Euclídea: `d(a,b) = √(81+81) ≈ 12.73` (¡enorme!),
`d(a,c) = 1`. La euclídea diría que `a` se parece a `c`; el coseno, que `a ≡ b`. Si la
norma solo refleja frecuencia del término, `b` es "la misma palabra dicha más veces" y el
coseno acierta.

**Ejercicio 3.** `rey − hombre + mujer = (1,1) − (1,0) + (0,1) = (0, 2) = reina`
exactamente (juguete diseñado para ello). `cos((0,2), reina) = 1`. En modelos reales la
precaución es doble: la regularidad es **aproximada** y las evaluaciones clásicas excluían
los términos de la consulta — sin esa exclusión, el vecino más cercano de la resta suele
ser `rey` mismo.

**Ejercicio 4.** `cos(gato, perro) ≈ 0.889`, `cos(gato, coche) = 0.0`,
`cos(gato, pez) = 7/(3·√6) ≈ 0.952` → el vecino más cercano es **pez**: comparte dirección
casi por completo pese a tener otra norma.


In [ ]:
result = run_lab("retrieval", seed=66)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 4 — coseno y vecino más cercano verificados con código
import math

def cos(u, v):
    dot = sum(a * b for a, b in zip(u, v))
    nu = math.sqrt(sum(a * a for a in u))
    nv = math.sqrt(sum(b * b for b in v))
    return dot / (nu * nv)

vectores = {"perro": (2, 1, 2), "coche": (2, 0, -1), "pez": (1, 2, 1)}
gato = (1, 2, 2)
sims = {k: round(cos(gato, v), 3) for k, v in vectores.items()}
print(sims)
print("vecino más cercano:", max(sims, key=sims.get))  # pez


In [ ]:
# Ejercicios 1-3 — verificación numérica
u, v, w = (2, 2, 1), (1, 2, 2), (-2, -2, -1)
print("cos(u,v):", round(cos(u, v), 3))   # 0.889
print("cos(u,w):", round(cos(u, w), 3))   # -1.0

a, b, c = (1, 1), (10, 10), (1, 0)
print("cos(a,b):", round(cos(a, b), 3), "cos(a,c):", round(cos(a, c), 3))

rey, hombre, mujer, reina = (1, 1), (1, 0), (0, 1), (0, 2)
q = tuple(r - h + m for r, h, m in zip(rey, hombre, mujer))
print("rey - hombre + mujer =", q, "cos con reina:", round(cos(q, reina), 3))


## Reflexión

1. El laboratorio `retrieval` ordena documentos por similitud. ¿Qué diferencia hay entre
   "el documento más similar" y "el documento que responde la pregunta", y qué evaluación
   detectaría la brecha?
2. Si usas embeddings para filtrar CV por similitud con "los perfiles que ya contratamos",
   ¿qué sesgo estás automatizando y qué mediría una auditoría seria?
3. ¿Por qué actualizar el modelo de embeddings de un buscador en producción exige reindexar
   todos los documentos, y qué pasaría si no lo haces?
